# Data Download
Put the SampleSubmission.csv, Test.csv, and Train.csv under the same directory
I put it in /Data - Lucky


In [1]:
#Import some libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pylab import rcParams
import seaborn as sb
sb.set_style('darkgrid')
rcParams['figure.figsize'] = 8,8
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
#import data
train = pd.read_csv('Data/Train.csv')
test=  pd.read_csv('Data/Test.csv')
submission = pd.read_csv('Data/SampleSubmission.csv')

In [3]:
train.columns, test.columns

(Index(['user_id', 'REGION', 'TENURE', 'MONTANT', 'FREQUENCE_RECH', 'REVENUE',
        'ARPU_SEGMENT', 'FREQUENCE', 'DATA_VOLUME', 'ON_NET', 'ORANGE', 'TIGO',
        'ZONE1', 'ZONE2', 'MRG', 'REGULARITY', 'TOP_PACK', 'FREQ_TOP_PACK',
        'CHURN'],
       dtype='str'),
 Index(['user_id', 'REGION', 'TENURE', 'MONTANT', 'FREQUENCE_RECH', 'REVENUE',
        'ARPU_SEGMENT', 'FREQUENCE', 'DATA_VOLUME', 'ON_NET', 'ORANGE', 'TIGO',
        'ZONE1', 'ZONE2', 'MRG', 'REGULARITY', 'TOP_PACK', 'FREQ_TOP_PACK'],
       dtype='str'))

# The Problem
(Taken from https://zindi.world/competitions/expresso-churn-prediction)


Expresso is an African telecommunications company that provides customers with airtime and mobile data bundles. The objective of this challenge is to develop a machine learning model to predict the likelihood of each Expresso customer “churning,” i.e. becoming inactive and not making any transactions for 90 days.

This solution will help Expresso to better serve their customers by understanding which customers are at risk of leaving.

Variable Definition Tables
| Variable | Description | Data type |
|---|---|---|
| REGION | Client location | object |
| TENURE | Duration in the network | object |
| MONTANT | Top-up amount | float64 |
| FREQUENCE_RECH | Number of times the customer refilled | float64 |
| REVENUE | Monthly revenue generated by the client | float64 |
| ARPU_SEGMENT | Revenue over 90 days / 3 | float64 |
| FREQUENCE | Number of times the client generated revenue | float64 |
| DATA_VOLUME | Number of connections | float64 |
| ON_NET | Calls within Expresso | float64 |
| ORANGE | Calls to Orange | float64 |
| TIGO | Calls to Tigo | float64 |
| ZONE1 | Calls to zone 1 | float64 |
| ZONE2 | Calls to zone 2 | float64 |
| MRG | Unclear definition; constant column | object |
| REGULARITY | Number of times the client is active over 90 days | int64 |
| TOP_PACK | Client’s most frequently activated pack | object |
| FREQ_TOP_PACK | Number of times the client activated the top pack | float64 |
| CHURN | Variable to predict (target) | int64 |

Details:
1. Income*: refer to how much revenue did the customer generate for the company
2. MRG^: Unclearly defined variable, after checking, it is just a constant column.

In [4]:
target = "CHURN"
X, y = train.copy(), train[target].copy()

# Drop constant columns
# dropping user_id as well for obvious reasons
# NOT Dropping target YET, just to simplify deduplication and other stuff down the line
X.drop(columns=['user_id', 'MRG'], inplace=True)


## Data Understanding

### Handling Missing Values

In [5]:
# Count missing values per column
missing_counts = X.isna().sum()
has_na_cols = missing_counts > 0

missing_counts[has_na_cols]

REGION             849299
MONTANT            756739
FREQUENCE_RECH     756739
REVENUE            726048
ARPU_SEGMENT       726048
FREQUENCE          726048
DATA_VOLUME       1060433
ON_NET             786675
ORANGE             895248
TIGO              1290016
ZONE1             1984327
ZONE2             2017224
TOP_PACK           902594
FREQ_TOP_PACK      902594
dtype: int64

In [6]:
cols_with_na = [
    'REGION',
    'MONTANT',
    'FREQUENCE_RECH',
    'REVENUE',
    'ARPU_SEGMENT',
    'FREQUENCE',
    'DATA_VOLUME',
    'ON_NET',
    'ORANGE',
    'TIGO',
    'ZONE1',
    'ZONE2',
    'TOP_PACK',
    'FREQ_TOP_PACK'
]

# Helper function
def summarize_against_label(df, col, label):
    missing = df[col].isna()
    summary = (
        pd.DataFrame({
            f'{col}_is_missing': missing,
            'CHURN': label
        }).groupby(f"{col}_is_missing")['CHURN']
        .agg(customer="size", churn_rate="mean")
    )
    return summary

# Check one by one, kinda manual, but, eh, unescapable
# Purpose: Find whether "NaN" has a meaning or actually missing
for col in cols_with_na:
    summary = summarize_against_label(X, col, y)
    print(summary)
    print("=" * 50)

# Manually check: Essentially, EVERY NA columns has some association with churn rate
# Need to validate later, but, keeping every cols for now.
na_cols_with_importance = [
    'REGION',
    'MONTANT',
    'FREQUENCE_RECH',
    'REVENUE',
    'ARPU_SEGMENT',
    'FREQUENCE',
    'DATA_VOLUME',
    'ON_NET',
    'ORANGE',
    'TIGO',
    'ZONE1', # Very many missing values, has a 0.2 correlation rate with CHURN
    'ZONE2', # Same with ZONE1
    'TOP_PACK',
    'FREQ_TOP_PACK'
]

                   customer  churn_rate
REGION_is_missing                      
False               1304749    0.018020
True                 849299    0.447987
                    customer  churn_rate
MONTANT_is_missing                      
False                1397309    0.050035
True                  756739    0.441463
                           customer  churn_rate
FREQUENCE_RECH_is_missing                      
False                       1397309    0.050035
True                         756739    0.441463
                    customer  churn_rate
REVENUE_is_missing                      
False                1428000    0.054057
True                  726048    0.450098
                         customer  churn_rate
ARPU_SEGMENT_is_missing                      
False                     1428000    0.054057
True                       726048    0.450098
                      customer  churn_rate
FREQUENCE_is_missing                      
False                  1428000    0.054057
True   

In [7]:
# is 0 used for numeric columns?
num_cols = X.select_dtypes(include=np.number).columns
for col in num_cols:
    print(f"Column {col}: {X[col].describe()['min']}")

Column MONTANT: 10.0
Column FREQUENCE_RECH: 1.0
Column REVENUE: 1.0
Column ARPU_SEGMENT: 0.0
Column FREQUENCE: 1.0
Column DATA_VOLUME: 0.0
Column ON_NET: 0.0
Column ORANGE: 0.0
Column TIGO: 0.0
Column ZONE1: 0.0
Column ZONE2: 0.0
Column REGULARITY: 1.0
Column FREQ_TOP_PACK: 1.0
Column CHURN: 0.0


In [8]:
# Deal with Missing Values
# Since each columns has some level of association:
# Numeric column -> 0
# We can use 0 here because it does indicate the user has no activity in that area, which is the assumption
# Categorical column - None
# No need for ColumnTransformer, as these are constant replacement of NaNs

# I know this includes the target column, but, no Na values in the target column, so, no worries
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

num_cols, cat_cols
X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna("None")

In [9]:
# No More NaN values!
missing_counts = X.isna().sum()
has_na_cols = missing_counts > 0

missing_counts[has_na_cols]

Series([], dtype: int64)

### Any Duplicates?


In [11]:
print(f"Duplicated rows count = {X.duplicated().sum()}")
# Note -> YES, A lot of duplications. But, this are different customers...
# How does each "duplication group" compare with the churn label?

# Exact duplicates include the label; feature duplicates exclude it.
total_duplicates = X.duplicated().sum()
feature_cols = X.columns.drop(target).tolist()
duplicate_mask = X.duplicated(subset=feature_cols, keep=False)
duplicate_rows = X.loc[duplicate_mask].copy()

# Count both labels for each repeated feature combination.
duplicate_summary = (
    duplicate_rows.groupby(feature_cols, dropna=False, observed=True, sort=False)[target]
    .agg(rows="size", distinct_labels="nunique", churn_1="sum")
)
duplicate_summary["churn_0"] = duplicate_summary["rows"] - duplicate_summary["churn_1"]
conflicting_groups = duplicate_summary.loc[
    duplicate_summary["distinct_labels"] > 1
].sort_values("rows", ascending=False)

print(f"Rows in repeated feature groups (including first occurrences): {len(duplicate_rows):,}")
print(f"Repeated feature groups: {len(duplicate_summary):,}")
print(f"Groups with consistent labels: {(duplicate_summary['distinct_labels'] == 1).sum():,}")
print(f"Groups with conflicting labels: {len(conflicting_groups):,}")
print(f"Rows in conflicting groups: {conflicting_groups['rows'].sum():,}")

# Each displayed group has identical features but both CHURN=0 and CHURN=1.
display(conflicting_groups.head(10).reset_index())


Duplicated rows count = 661897
Rows in repeated feature groups (including first occurrences): 692,284
Repeated feature groups: 21,753
Groups with consistent labels: 13,119
Groups with conflicting labels: 8,634
Rows in conflicting groups: 633,275


,REGION,TENURE,MONTANT,FREQUENCE_RECH,REVENUE,ARPU_SEGMENT,FREQUENCE,DATA_VOLUME,ON_NET,ORANGE,TIGO,ZONE1,ZONE2,REGULARITY,TOP_PACK,FREQ_TOP_PACK,rows,distinct_labels,churn_1,churn_0
0,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,None,0.0,133708,2,105059,28649
1,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,None,0.0,72200,2,49960,22240
2,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,None,0.0,45608,2,28196,17412
3,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,None,0.0,31621,2,18087,13534
4,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,None,0.0,23153,2,12080,11073
5,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,None,0.0,17679,2,8544,9135
6,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,None,0.0,13980,2,6256,7724
7,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8,None,0.0,10968,2,4691,6277
8,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,None,0.0,8761,2,3538,5223
9,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,None,0.0,7448,2,2827,4621
